# Авито: CatBoost-ранкер v3 — кандидаты BM25 + энкодер, признаки косинуса

**Входы (Add Input):**
- вывод ноутбука признаков v2 (`features_v2/`: `train.parquet`, `val.parquet`, `bench.parquet`, `features_v2.json`)
- `microcat_proba.parquet` от классификатора (твой датасет `new_features_for_catboost`)

**Что делает:**
1. Присоединяет признаки классификатора микрокатегорий (как в v2)
2. Обучает **две модели на одних и тех же кандидатах**: без признаков энкодера и с ними (`cos_full`, `cos_title`, `rank_dense`, `cos_gap`)
3. Считает Recall@50 на train и val, потолок, Recall@K, важность признаков
4. **Проверяет утечку**: если модель с косинусом выросла на train заметно сильнее, чем на val, признак на train «подсказывает» ответы
5. (`MAKE_ANSWER = True`) Предсказывает бенчмарк и пишет `answer.csv` в формате задания с проверками всех требований

GPU: одной T4 достаточно.

### Импорты и параметры

In [1]:
import os, glob, json, time, gc, warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

SEED = 42
K = 50
ES_SHARE = 0.1               # доля train-запросов для early stopping

# Явно бесполезные признаки (см. обсуждение):
#   geo_known       — константа (1 у всех)
#   rating_ok       — почти всегда 1 и уже учтён в filter_ok
#   filter_text_ok  — дублирует filter_ok
#   search_category — в train почти нет значения 0, модель не научится; категория уже учтена правилом при сборке пула
DROP_FEATURES = ['geo_known', 'rating_ok', 'filter_text_ok', 'search_category']
# по итогам абляции v1 (без них val Recall@50 0.8778 против 0.8785 — в пределах шума):
DROP_FEATURES += ['item_is_message_forbidden', 'rating_isna', 'price_is_symbolic', 'item_is_phone_hidden',
                  'q_n_terms', 'q_len_words', 'rank_global']
DROP_FEATURES += ['mc_share_top20']       # после появления классификатора почти нулевая польза
MC_FEATURES = ['mc_proba', 'mc_proba_rel', 'mc_top1_proba']   # mc_rank убран: дублирует mc_proba
MC_PATH_HINT = 'catboost'   # подстрока пути к НУЖНОМУ microcat_proba.parquet (у тебя датасет new_features_for_catboost)
MAKE_ANSWER = True          # собрать answer.csv для бенчмарка

TASK_TYPE = 'GPU' if get_gpu_device_count() > 0 else 'CPU'
print('обучение на:', TASK_TYPE)

обучение на: GPU


## 1. Загрузка

Все файлы ищутся в подключённых входах. Если найдено несколько `microcat_proba.parquet`, берётся тот, в пути которого есть `MC_PATH_HINT`.

In [2]:
def find_all(name):
    return [p for root in ['/kaggle/input', '/kaggle/working', '.']
            for p in sorted(glob.glob(os.path.join(root, '**', name), recursive=True))]

hits = find_all('features_v2.json')
assert hits, 'features_v2.json не найден: подключи вывод ноутбука признаков v2'
FEAT_DIR = os.path.dirname(hits[0])
meta = json.load(open(f'{FEAT_DIR}/features_v2.json'))
COS_FEATURES = meta['cos_features']
train_df = pd.read_parquet(f'{FEAT_DIR}/train.parquet')
val_df = pd.read_parquet(f'{FEAT_DIR}/val.parquet')
bench_df = pd.read_parquet(f'{FEAT_DIR}/bench.parquet') if MAKE_ANSWER else None

# --- признаки классификатора микрокатегорий
mc_hits = find_all('microcat_proba.parquet')
assert mc_hits, 'microcat_proba.parquet не найден: подключи вывод классификатора'
print('найдены файлы классификатора:', mc_hits)
MC_PATH = next((p for p in mc_hits if MC_PATH_HINT in p), mc_hits[0])
print('классификатор:', MC_PATH)
mc = pd.read_parquet(MC_PATH)
MC_TOP_N = mc.groupby('qid').size().max()
mc['mc_rank'] = (mc.groupby('qid')['proba'].rank(method='first', ascending=False) - 1).astype('float32')
mc_top1 = mc.groupby('qid')['proba'].max().rename('mc_top1_proba')
mc = mc.rename(columns={'microcat': 'item_microcat_id', 'proba': 'mc_proba'})

def add_mc_features(df):
    """Присоединяет вероятность микрокатегории кандидата; микрокатегория вне топа -> proba 0, rank = MC_TOP_N."""
    df = df.merge(mc, on=['qid', 'item_microcat_id'], how='left')
    df = df.merge(mc_top1, left_on='qid', right_index=True, how='left')
    df['mc_proba'] = df['mc_proba'].fillna(0).astype('float32')
    df['mc_rank'] = df['mc_rank'].fillna(MC_TOP_N).astype('float32')
    df['mc_proba_rel'] = (df['mc_proba'] / df['mc_top1_proba'].clip(lower=1e-6)).astype('float32')
    return df

for name in ['train_df', 'val_df'] + (['bench_df'] if MAKE_ANSWER else []):
    df = globals()[name]
    cover = df.qid.drop_duplicates().isin(set(mc.qid)).mean()
    print(f'{name}: предсказания классификатора есть у {cover:.2%} запросов')
    if cover < 0.999:
        print(f'  ВНИМАНИЕ: у {1 - cover:.2%} запросов {name} нет предсказаний классификатора — признаки mc у них будут нулями. '
              f'Скорее всего, подключён не тот microcat_proba.parquet (см. MC_PATH_HINT)')
    globals()[name] = add_mc_features(df)

FEATURES_BASE = [f for f in meta['features'] if f not in DROP_FEATURES + COS_FEATURES] + MC_FEATURES   # без энкодера
FEATURES = FEATURES_BASE + COS_FEATURES                                                                  # с энкодером
CAT_FEATURES = [f for f in meta['cat_features'] if f in FEATURES]
for df in [d for d in (train_df, val_df, bench_df) if d is not None]:
    for c in CAT_FEATURES:
        df[c] = df[c].astype('int64').astype(str)      # CatBoost ждёт категориальные как строки/целые
    # CatBoost требует, чтобы строки одной группы (запроса) шли подряд
    df.sort_values('qid', kind='stable', inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f'папка: {FEAT_DIR}')
print(f'train: {train_df.qid.nunique():,} запросов, {len(train_df):,} строк')
print(f'val:   {val_df.qid.nunique():,} запросов, {len(val_df):,} строк')
print(f'признаков: без энкодера {len(FEATURES_BASE)}, с энкодером {len(FEATURES)} | категориальные: {CAT_FEATURES}')

найдены файлы классификатора: ['/kaggle/input/datasets/olegkaran/new-features-for-catboost/microcat_proba.parquet']
классификатор: /kaggle/input/datasets/olegkaran/new-features-for-catboost/microcat_proba.parquet
train_df: предсказания классификатора есть у 100.00% запросов
val_df: предсказания классификатора есть у 100.00% запросов
bench_df: предсказания классификатора есть у 100.00% запросов
папка: /kaggle/input/notebooks/olegkaran/full-features-v2/features_v2
train: 15,000 запросов, 4,269,507 строк
val:   3,000 запросов, 857,038 строк
признаков: без энкодера 26, с энкодером 30 | категориальные: ['item_microcat_id']


## 2. Метрика и бейзлайны

- `recall_at_k(df, score)`: для каждого запроса берём топ-K по скору, считаем `сумма target / n_pos_total`, усредняем
- **BM25-бейзлайн**: сначала объявления пула, внутри пула по `bm25_all`, то же самое, что «топ-50 BM25 в пуле» из ноутбука признаков
- **потолок**: если бы модель ставила все релевантные кандидаты наверх

In [3]:
def recall_at_k(df, score, k=K):
    d = pd.DataFrame({'qid': df['qid'].values, 'target': df['target'].values,
                      'n_pos': df['n_pos_total'].values, 'score': np.asarray(score)})
    d['rank'] = d.groupby('qid')['score'].rank(method='first', ascending=False)
    hits = d.loc[d['rank'] <= k].groupby('qid')['target'].sum()
    n_pos = d.groupby('qid')['n_pos'].first()
    return float((hits.reindex(n_pos.index, fill_value=0) / n_pos).mean())

def bm25_baseline_score(df):
    return df['in_pool'].values * 1e6 + df['bm25_all'].values

def dense_baseline_score(df):
    return df['in_pool'].values * 1e6 + df['cos_full'].values

def ceiling(df):
    return recall_at_k(df, df['target'].values, k=10**9)

RESULTS = {}
for name, df in [('train', train_df), ('val', val_df)]:
    RESULTS[(name, 'потолок (все кандидаты)')] = ceiling(df)
    RESULTS[(name, 'BM25 в пуле')] = recall_at_k(df, bm25_baseline_score(df))
    RESULTS[(name, 'энкодер в пуле')] = recall_at_k(df, dense_baseline_score(df))
pd.Series(RESULTS).unstack(0)

,train,val
BM25 в пуле,0.871443,0.836222
потолок (все кандидаты),0.971916,0.969333
энкодер в пуле,0.886240,0.841222


## 3. Early-stopping выборка

10% train-запросов откладываем для выбора числа деревьев. Если использовать для этого val, итоговая оценка на val будет чуть завышена.

Из обучающей части убираем запросы, где среди кандидатов нет ни одного релевантного: ранжирующему лоссу на них нечего сравнивать,
они только тратят время. На оценку это не влияет, в метриках такие запросы остаются.

In [4]:
rng = np.random.default_rng(SEED)
tr_qids = train_df.qid.unique()
es_qids = set(rng.choice(tr_qids, size=int(len(tr_qids) * ES_SHARE), replace=False))
is_es = train_df.qid.isin(es_qids).values
fit_df, es_df = train_df[~is_es], train_df[is_es]

has_pos = fit_df.groupby('qid')['target'].transform('max') > 0
print(f'обучение: {fit_df.qid.nunique():,} запросов, из них с релевантным в кандидатах: {fit_df.loc[has_pos, "qid"].nunique():,}')
fit_df = fit_df[has_pos]
# в early-stopping выборке тоже оставляем только запросы с релевантным в кандидатах:
# встроенный RecallAt в CatBoost засчитывает пустые группы как успех, и метрика «залипает» около 1
es_df = es_df[es_df.groupby('qid')['target'].transform('max') > 0]
print(f'early stopping: {es_df.qid.nunique():,} запросов')
assert len(fit_df) and len(es_df), 'пустая обучающая или early-stopping выборка — проверь файлы признаков'

def make_pool(df, features):
    cats = [c for c in CAT_FEATURES if c in features]
    return Pool(df[features], label=df['target'].values, group_id=df['qid'].values, cat_features=cats)

обучение: 13,500 запросов, из них с релевантным в кандидатах: 13,139
early stopping: 1,463 запросов


## 4. Обучение: две модели (без и с признаками энкодера)

- `YetiRank`: ранжирующий лосс, оптимизирует порядок внутри запроса, а не вероятность класса. Это то, что нужно для топ-50
- `eval_metric = RecallAt:top=50`: по нему выбирается лучшая итерация. Считается по кандидатам, поэтому чуть выше «нашего» Recall (если на GPU он не поддерживается, ноутбук сам переключится на NDCG@50)
- early stopping: остановка, если метрика не улучшается 200 итераций

In [5]:
PARAMS = dict(
    loss_function='YetiRank',
    eval_metric='RecallAt:top=50',
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    random_seed=SEED,
    task_type=TASK_TYPE,
    early_stopping_rounds=200,
    use_best_model=True,
    verbose=200,
)
if TASK_TYPE == 'GPU':
    PARAMS['devices'] = '0'

def train_ranker(features, fit, es, params=PARAMS):
    t = time.time()
    try:
        model = CatBoostRanker(**params)
        model.fit(make_pool(fit, features), eval_set=make_pool(es, features))
    except Exception as e:
        # подстраховка: если в этой версии CatBoost метрика RecallAt не поддерживается на GPU, берём NDCG@50
        print(f'RecallAt не сработал ({str(e)[:120]}), переключаюсь на NDCG:top=50')
        model = CatBoostRanker(**{**params, 'eval_metric': 'NDCG:top=50'})
        model.fit(make_pool(fit, features), eval_set=make_pool(es, features))
    print(f'обучено за {time.time() - t:.0f} с, лучшая итерация: {model.get_best_iteration()}')
    return model

print('=== модель БЕЗ признаков энкодера ===')
model_base = train_ranker(FEATURES_BASE, fit_df, es_df)
print('\n=== модель С признаками энкодера ===')
model = train_ranker(FEATURES, fit_df, es_df)

=== модель БЕЗ признаков энкодера ===
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, RecallAt is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.8919483	test: 0.8986842	best: 0.8986842 (0)	total: 739ms	remaining: 36m 56s
200:	learn: 0.9498710	test: 0.9603782	best: 0.9603782 (188)	total: 1m 26s	remaining: 20m 10s
400:	learn: 0.9522780	test: 0.9634769	best: 0.9640237 (303)	total: 2m 56s	remaining: 19m 1s
600:	learn: 0.9551874	test: 0.9675780	best: 0.9675780 (599)	total: 4m 26s	remaining: 17m 44s
800:	learn: 0.9570240	test: 0.9691729	best: 0.9691729 (680)	total: 5m 57s	remaining: 16m 22s
1000:	learn: 0.9580383	test: 0.9712235	best: 0.9712235 (860)	total: 7m 29s	remaining: 14m 57s
bestTest = 0.9712235133
bestIteration = 860
Shrink model to first 861 iterations.
обучено за 483 с, лучшая итерация: 860

=== модель С признаками энкодера ===
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, RecallAt is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric RecallAt:top=50 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.9250629	test: 0.9281841	best: 0.9281841 (0)	total: 575ms	remaining: 28m 44s
200:	learn: 0.9642524	test: 0.9762930	best: 0.9762930 (175)	total: 1m 30s	remaining: 21m
400:	learn: 0.9662947	test: 0.9786853	best: 0.9786853 (308)	total: 3m 1s	remaining: 19m 39s
600:	learn: 0.9678350	test: 0.9819891	best: 0.9819891 (595)	total: 4m 33s	remaining: 18m 12s
800:	learn: 0.9698360	test: 0.9825359	best: 0.9825359 (767)	total: 6m 5s	remaining: 16m 42s
1000:	learn: 0.9717374	test: 0.9845865	best: 0.9845865 (960)	total: 7m 36s	remaining: 15m 10s
bestTest = 0.9845864662
bestIteration = 960
Shrink model to first 961 iterations.
обучено за 535 с, лучшая итерация: 960


## 5. Результаты: Recall@50

In [6]:
pred = {'train': model.predict(train_df[FEATURES]), 'val': model.predict(val_df[FEATURES])}
pred_base = {'train': model_base.predict(train_df[FEATURES_BASE]), 'val': model_base.predict(val_df[FEATURES_BASE])}
for name, df in [('train', train_df), ('val', val_df)]:
    RESULTS[(name, 'CatBoost без энкодера')] = recall_at_k(df, pred_base[name])
    RESULTS[(name, 'CatBoost + энкодер')] = recall_at_k(df, pred[name])

res = pd.Series(RESULTS).unstack(0)[['train', 'val']]
res = res.loc[['BM25 в пуле', 'энкодер в пуле', 'CatBoost без энкодера', 'CatBoost + энкодер', 'потолок (все кандидаты)']]
display(res.style.format('{:.4f}'))
v, t = res['val'], res['train']
print(f'прирост от энкодера на val: {v["CatBoost + энкодер"] - v["CatBoost без энкодера"]:+.4f}  (изменения меньше ~0.003 — шум)')
print(f'закрыто разрыва BM25 -> потолок: без энкодера {(v["CatBoost без энкодера"] - v["BM25 в пуле"]) / (v["потолок (все кандидаты)"] - v["BM25 в пуле"]):.1%}, '
      f'с энкодером {(v["CatBoost + энкодер"] - v["BM25 в пуле"]) / (v["потолок (все кандидаты)"] - v["BM25 в пуле"]):.1%}')

# Проверка утечки: прирост на train не должен заметно превышать прирост на val
d_tr = t['CatBoost + энкодер'] - t['CatBoost без энкодера']
d_val = v['CatBoost + энкодер'] - v['CatBoost без энкодера']
print(f'прирост от энкодера: train {d_tr:+.4f}, val {d_val:+.4f} -> '
      f'{"утечки не видно" if d_tr - d_val < 0.01 else "на train прирост заметно больше — косинус может «подсказывать» на train"}')

,train,val
BM25 в пуле,0.8714,0.8362
энкодер в пуле,0.8862,0.8412
CatBoost без энкодера,0.9321,0.8971
CatBoost + энкодер,0.9456,0.9284
потолок (все кандидаты),0.9719,0.9693


прирост от энкодера на val: +0.0313  (изменения меньше ~0.003 — шум)
закрыто разрыва BM25 -> потолок: без энкодера 45.7%, с энкодером 69.2%
прирост от энкодера: train +0.0135, val +0.0313 -> утечки не видно


In [7]:
# Recall при разных K: насколько круто растёт кривая; если @100 намного выше @50, модель «почти» находит ответы
curve = pd.DataFrame({k: {'BM25': recall_at_k(val_df, bm25_baseline_score(val_df), k),
                          'энкодер': recall_at_k(val_df, dense_baseline_score(val_df), k),
                          'CatBoost без энкодера': recall_at_k(val_df, pred_base['val'], k),
                          'CatBoost + энкодер': recall_at_k(val_df, pred['val'], k)} for k in [10, 20, 50, 100, 200]})
print('val: Recall@K')
display(curve.style.format('{:.4f}'))

val: Recall@K


,10,20,50,100,200
BM25,0.6225,0.7356,0.8362,0.8782,0.9037
энкодер,0.6142,0.7386,0.8412,0.8929,0.9346
CatBoost без энкодера,0.7107,0.8069,0.8971,0.9410,0.9615
CatBoost + энкодер,0.7624,0.8483,0.9284,0.9575,0.9677


## 6. Важность признаков (модель с энкодером)

- `PredictionValuesChange`: насколько признак в среднем меняет предсказание (быстро, но «важный» не значит «полезный»)
- `LossFunctionChange` на val: насколько **ухудшается лосс на val**, если признак убрать. Ноль или минус — кандидат в мусор

In [8]:
val_pool = make_pool(val_df, FEATURES)
imp = pd.DataFrame({
    'PredictionValuesChange': model.get_feature_importance(type='PredictionValuesChange'),
    'LossFunctionChange_val': model.get_feature_importance(val_pool, type='LossFunctionChange'),
}, index=FEATURES).sort_values('LossFunctionChange_val', ascending=False)
display(imp.style.format('{:.4f}').background_gradient(cmap='RdYlGn', subset=['LossFunctionChange_val']))
print('признаки с нулевой/отрицательной пользой на val:', imp.index[imp['LossFunctionChange_val'] <= 0].tolist())

,PredictionValuesChange,LossFunctionChange_val
rank_pool,8.1321,0.0398
cov_all,15.7906,0.0322
cos_gap,21.6799,0.0111
same_loc,7.4471,0.0106
dist_km,20.5674,0.0100
rank_dense,7.7510,0.0091
reviews_log,3.4904,0.0089
bm25_rel,2.8174,0.0057
filter_ok,2.2786,0.0041
loc_n_items,3.4166,0.0017


признаки с нулевой/отрицательной пользой на val: ['in_pool', 'bm25_params', 'cos_full']


## 7. Сохранение модели и предсказаний на val

In [9]:
OUT = '/kaggle/working/catboost' if os.path.isdir('/kaggle/working') else './catboost'
os.makedirs(OUT, exist_ok=True)
model.save_model(f'{OUT}/ranker_enc.cbm')
model_base.save_model(f'{OUT}/ranker_no_enc.cbm')
val_df[['qid', 'item_id', 'target']].assign(score=pred['val']).to_parquet(f'{OUT}/val_pred.parquet', index=False)
json.dump({'features': FEATURES, 'cat_features': CAT_FEATURES, 'drop': DROP_FEATURES,
           'params': {k: v for k, v in PARAMS.items() if k != 'verbose'},
           'val_recall50': res.loc['CatBoost + энкодер', 'val'], 'val_recall50_base': res.loc['CatBoost без энкодера', 'val']}, open(f'{OUT}/model_meta.json', 'w'), ensure_ascii=False, indent=2)
print(os.listdir(OUT))

['ranker_enc.cbm', 'model_meta.json', 'ranker_no_enc.cbm', 'val_pred.parquet']


## 8. answer.csv для бенчмарка

Лучшая по val модель ранжирует кандидатов каждого запроса бенчмарка, топ-50 `item_id` записываются через пробел.
Перед сохранением проверяются все требования задания: 2452 строки без повторов, ровно две колонки,
`query_id` из 16 символов, не больше 50 уникальных `item_id` в строке, каждый `item_id` из 16 символов `0-9a-f`.

In [10]:
if MAKE_ANSWER:
    # берём модель, которая лучше на val
    use_enc = res.loc['CatBoost + энкодер', 'val'] >= res.loc['CatBoost без энкодера', 'val']
    best_model, best_feats = (model, FEATURES) if use_enc else (model_base, FEATURES_BASE)
    print(f'ответ строит модель {"С энкодером" if use_enc else "БЕЗ энкодера"} '
          f'(val Recall@50 {max(res.loc["CatBoost + энкодер", "val"], res.loc["CatBoost без энкодера", "val"]):.4f})')
    b = bench_df[['qid', 'item_id']].assign(score=best_model.predict(bench_df[best_feats]))
    b = b.sort_values(['qid', 'score'], ascending=[True, False]).drop_duplicates(['qid', 'item_id'])
    top = b.groupby('qid').head(K)
    answer = (top.groupby('qid', sort=False)['item_id'].apply(lambda s: ' '.join(s)).reset_index()
              .rename(columns={'qid': 'query_id', 'item_id': 'answer'}))

    # --- проверки формата
    assert answer.columns.tolist() == ['query_id', 'answer']
    assert answer.query_id.is_unique, 'повторы query_id'
    assert answer.query_id.str.len().eq(16).all(), 'query_id не из 16 символов'
    ids = answer.answer.str.split(' ')
    assert ids.map(len).le(K).all(), 'больше 50 item_id в строке'
    assert ids.map(lambda x: len(set(x)) == len(x)).all(), 'повторы item_id внутри строки'
    assert ids.explode().str.fullmatch(r'[0-9a-f]{16}').all(), 'item_id не в формате 16 символов 0-9a-f'
    print(f'запросов в ответе: {len(answer):,} (ожидается 2452) | item_id на запрос: '
          f'мин {ids.map(len).min()}, медиана {ids.map(len).median():.0f}')
    if len(answer) != 2452:
        print('ВНИМАНИЕ: число строк не 2452 — проверь bench.parquet')
    answer.to_csv(f'{OUT}/answer.csv', index=False)
    print(f'сохранено: {OUT}/answer.csv')
    display(answer.head(3))

ответ строит модель С энкодером (val Recall@50 0.9284)
запросов в ответе: 2,452 (ожидается 2452) | item_id на запрос: мин 50, медиана 50
сохранено: /kaggle/working/catboost/answer.csv


,query_id,answer
0,00WuFMaXSFZBxSzT,c90da827b4ad7b5a 75d44a1b48f4faab 980953f92323...
1,03ztb1gtRFC4K4vP,a7452b3fb8825c43 a5afb8923b2d1863 6a920b36f8ad...
2,052l5ySkk2Asxdni,3b414fcfcca8c33d b27a31ec69daa554 29d267e0809c...


In [11]:
# Проверка answer.csv по требованиям задания (читаем файл заново, как проверяющий)
import glob, os
import pandas as pd

def find_all(name):
    return [p for root in ['/kaggle/working', '/kaggle/input', '.']
            for p in sorted(glob.glob(os.path.join(root, '**', name), recursive=True))]

ANSWER_PATH = find_all('answer.csv')[0]
print('проверяю:', ANSWER_PATH)

chk = pd.read_csv(ANSWER_PATH, dtype=str, keep_default_na=False, encoding='utf-8')
ids = chk['answer'].str.split(' ')
flat = ids.explode()

checks = {
    'ровно две колонки query_id, answer':      chk.columns.tolist() == ['query_id', 'answer'],
    'query_id без повторов':                   chk['query_id'].is_unique,
    'query_id ровно 16 символов':              chk['query_id'].str.len().eq(16).all(),
    'в каждой строке от 1 до 50 item_id':      ids.map(len).between(1, 50).all(),
    'нет повторов item_id внутри строки':      ids.map(lambda x: len(set(x)) == len(x)).all(),
    'item_id — 16 символов 0-9a-f':            flat.str.fullmatch(r'[0-9a-f]{16}').all(),
}

raw_q, raw_i = find_all('benchmark_queries.parquet'), find_all('benchmark_items.parquet')
if raw_q:
    expected = set(pd.read_parquet(raw_q[0], columns=['query_id'])['query_id'])
    got = set(chk['query_id'])
    checks['все query_id бенчмарка есть, лишних нет'] = (got == expected)
    print(f'нет в ответе: {len(expected - got)}, лишних: {len(got - expected)}')
else:
    checks['строк ровно 2452 (исходный файл не подключён)'] = (len(chk) == 2452)
if raw_i:
    corpus = set(pd.read_parquet(raw_i[0], columns=['item_id'])['item_id'])
    checks['все item_id есть в benchmark_items'] = flat.isin(corpus).all()
else:
    print('benchmark_items.parquet не подключён: сверка item_id с корпусом пропущена')

for name, ok in checks.items():
    print(f'{"✓" if ok else "✗"} {name}')
print(f'\nстрок: {len(chk):,} | item_id на запрос: мин {ids.map(len).min()}, медиана {ids.map(len).median():.0f}, макс {ids.map(len).max()}')
print('ИТОГ:', 'файл соответствует требованиям' if all(checks.values()) else 'ЕСТЬ ОШИБКИ, см. ✗ выше')

print('\nпервые строки файла как есть:')
with open(ANSWER_PATH, encoding='utf-8') as f:
    for _ in range(3):
        print(f.readline().rstrip()[:150] + ' ...')

проверяю: /kaggle/working/catboost/answer.csv
нет в ответе: 0, лишних: 0
✓ ровно две колонки query_id, answer
✓ query_id без повторов
✓ query_id ровно 16 символов
✓ в каждой строке от 1 до 50 item_id
✓ нет повторов item_id внутри строки
✓ item_id — 16 символов 0-9a-f
✓ все query_id бенчмарка есть, лишних нет
✓ все item_id есть в benchmark_items

строк: 2,452 | item_id на запрос: мин 50, медиана 50, макс 50
ИТОГ: файл соответствует требованиям

первые строки файла как есть:
query_id,answer ...
00WuFMaXSFZBxSzT,c90da827b4ad7b5a 75d44a1b48f4faab 980953f92323e8d6 2f82afc9946e6c84 f61c2546f43fc8d8 c86f7a252955f4b1 f37f88b2bd8263bc 2102627765180b ...
03ztb1gtRFC4K4vP,a7452b3fb8825c43 a5afb8923b2d1863 6a920b36f8ad8241 f4a78af5d8e7a2b6 75b818b60d321d51 0214fddcf70fb7da 6001581316c66435 0f3d8dd771cfc8 ...
